!pip install langchain==1.2.12

python 3.13

!pip install langchain_community

# SummarizationMiddleware

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_community.retrievers import WikipediaRetriever
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from initialize_llm import initialize


initialize()

retriever = WikipediaRetriever(  # type: ignore
    top_k_results=1,
    doc_content_chars_max=20_000,
)

@tool
def wikipedia_tool(query: str):
    """Fetch content of Wikipedia page from top hit of a query."""
    result = retriever.invoke(query)
    if result:
        return result[0].page_content
    return "No Result found"

summary_prompt = """
Summarize the main thrust of this conversation. What have the human and assistant
discussed so far? Focus on key facts and requests.
<messages>
Messages to summarize:
{messages}
</messages>
"""

agent = create_agent(
    model="gpt-4o",
    tools=[wikipedia_tool],
    middleware=[
        SummarizationMiddleware(
            model="gpt-4.1-mini",
            summary_prompt=summary_prompt,
            # Trigger summarization when 70% of context is used. Langchain has configured LLM context length
            trigger=("fraction", 0.7),
            # Keep the most recent 30% of messages in full
            keep=("fraction", 0.3),
            # No additional trimming before summarization
            trim_tokens_to_summarize=None,
        )
    ]
)

response = agent.invoke({"messages": [HumanMessage(content="Share me G42 company details")]})

In [ ]:
response

In [ ]:
print(response['messages'][-1].content)

# Tool limit